# Kuhn Poker: learning to bluff 🃏
Kuhn poker has three cards (J, Q, K), one card each, one betting round. It's tiny but has everything that makes poker hard: hidden information and bluffing. **Counterfactual regret minimisation (CFR)**, the algorithm family behind superhuman poker bots like Libratus and Pluribus, learns the Nash equilibrium by self-play. And yes, it learns to **bluff with the jack**.

Background: [[RL Basics and MDPs]] (games), [[Probability for ML]] · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import itertools, numpy as np
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
CARDS = ["J", "Q", "K"]
class InfoSet:
    def __init__(self): self.regret_sum = np.zeros(2); self.strategy_sum = np.zeros(2)     # actions: p = pass/check/fold, b = bet/call
nodes = {}
def payoff(cards, history, player):
    """Terminal utility for `player` (the one to act next) or None if not terminal."""
    if history in ("pp", "bp", "pbp") or history.endswith("bb"):
        if history.endswith("p") and history != "pp": return 1                             # the opponent folded
        stake = 2 if history.endswith("bb") else 1
        return stake if cards[player] > cards[1 - player] else -stake
    return None

## 1. Regret matching
Play each action in proportion to its **positive** accumulated regret; if no action has positive regret, play uniformly.

In [ ]:
def regret_matching(regret_sum):
    # TODO 1: normalised positive regrets, or uniform
    raise NotImplementedError  # TODO 1

## 2. The CFR recursion
At a decision node (information set = my card + betting history): get the current strategy by regret matching, add it to `strategy_sum` weighted by **my** reach probability, recursively get the utility of each action (negated: utilities are from the next player's view), and add the **counterfactual regrets** `p_other · (action utility − node utility)` to `regret_sum`. Return the node utility. `p1, p2` are the reach probabilities of player 0 and 1.

In [ ]:
def cfr(cards, history, p1, p2):
    player = len(history) % 2
    u = payoff(cards, history, player)
    if u is not None: return u
    node = nodes.setdefault(CARDS[cards[player]] + history, InfoSet())
    p_self, p_other = (p1, p2) if player == 0 else (p2, p1)
    # TODO 2: strategy, strategy_sum update, action utilities, regret update, return node utility
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_cfr():
    return np.allclose(regret_matching(np.array([3.0, 1.0])), [0.75, 0.25]) and np.allclose(regret_matching(np.array([-1.0, -2.0])), [0.5, 0.5]) and \
        payoff((2, 0), "pp", 0) == 1 and payoff((0, 2), "bb", 0) == -2 and payoff((0, 2), "bp", 0) == 1
check("regret matching", _t_cfr, "Regrets [3, 1] → [0.75, 0.25]; all negative → uniform. (Payoffs are given.)")
if ready("regret matching"):
    try:
        value = 0.0; ITERS = 20000
        for i in range(ITERS):
            for cards in itertools.permutations(range(3), 2): value += cfr(cards, "", 1, 1) / 6
        avg = {k: n.strategy_sum / n.strategy_sum.sum() for k, n in sorted(nodes.items())}
        print(f"game value for player 1: {value / ITERS:+.4f}   (theory: −1/18 = {-1/18:+.4f})")
        for k, s in avg.items(): print(f"  {k:<4} pass/check {s[0]:.2f}   bet/call {s[1]:.2f}")
        alpha = avg["J"][1]
        print(f"\nbluff: player 1 bets with the jack {alpha:.0%} of the time; theory says bet the king {3 * alpha:.0%} (= 3α) → learned {avg['K'][1]:.0%}")
        check("Nash equilibrium", lambda: abs(value / ITERS + 1 / 18) < 0.005 and 0.0 < alpha < 0.34 and abs(avg["K"][1] - 3 * alpha) < 0.1 and avg["Qb"][1] < 0.4,
              "Game value ≈ −1/18, the jack is bet (bluffed) with prob. α ≤ 1/3, the king with ≈ 3α.")
    except NotImplementedError:
        print("❌ Nash equilibrium: not implemented yet – fill in the TODO above.")

**Reading the strategy:** the first player bluffs with the J a little, value-bets the K about three times as often, and never bets the Q. The second player calls a bet with the Q exactly 1/3 of the time, which makes player 1's bluffs break even. Nobody programmed "bluff": it falls out of minimising regret.

<details><summary>▶ Solution</summary>

```python
def regret_matching(regret_sum):
    pos = np.maximum(regret_sum, 0)
    return pos / pos.sum() if pos.sum() > 0 else np.full(len(regret_sum), 1 / len(regret_sum))
```

```python
    # inside cfr(), after p_self, p_other = ...
    strategy = regret_matching(node.regret_sum)
    node.strategy_sum += p_self * strategy
    utils = np.array([-cfr(cards, history + a, p1 * strategy[i] if player == 0 else p1, p2 * strategy[i] if player == 1 else p2) for i, a in enumerate("pb")])
    node_util = strategy @ utils
    node.regret_sum += p_other * (utils - node_util)
    return node_util
```
</details>

## Stretch goals
- Play against the learned strategy yourself (pick a card at random, choose actions).
- Leduc poker (6 cards, 2 rounds): same CFR, bigger tree.
- Plot the exploitability of the average strategy vs. iterations: CFR converges as 1/√T.